# Phase 1 — Bronze Layer: Environment Setup & Data Ingestion

Ingests all 9 raw Olist CSVs into the Bronze Delta layer with schema
inference and audit columns (`ingestion_timestamp`, `source_file`).

In [ ]:
from pyspark.sql import functions as F

## Config — raw data path

In [ ]:
dbutils.widgets.text("raw_path", "/Volumes/de_capstone/default/olist_data")

RAW_PATH = dbutils.widgets.get("raw_path")

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

## Dataset map

All 9 raw CSVs, not just the 3 named in the rubric brief (`orders`,
`customers`, `products`) — Gold's aggregates also need `order_items` and
`payments`.

In [ ]:
DATASETS = {
    "olist_orders_dataset.csv": "bronze.orders",
    "olist_customers_dataset.csv": "bronze.customers",
    "olist_products_dataset.csv": "bronze.products",
    "olist_order_items_dataset.csv": "bronze.order_items",
    "olist_order_payments_dataset.csv": "bronze.order_payments",
    "olist_order_reviews_dataset.csv": "bronze.order_reviews",
    "olist_sellers_dataset.csv": "bronze.sellers",
    "olist_geolocation_dataset.csv": "bronze.geolocation",
    "product_category_name_translation.csv": "bronze.product_category_translation",
}

## Ingest each CSV to a Bronze Delta table

In [ ]:
def ingest_to_bronze(file_name: str, table_name: str) -> None:
    source_path = f"{RAW_PATH}/{file_name}"

    df = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(source_path)
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
    )

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        table_name
    )

    count = spark.table(table_name).count()
    print(f"{table_name}: {count} rows ingested from {file_name}")


for file_name, table_name in DATASETS.items():
    ingest_to_bronze(file_name, table_name)

## Verify — schema + row counts

Prints the schema and row count for every Bronze table as a sanity check
after ingestion.

In [ ]:
for table_name in DATASETS.values():
    print(f"\n=== {table_name} ===")
    spark.table(table_name).printSchema()

In [ ]:
display(spark.sql("SHOW TABLES IN bronze"))